In [1]:
import os

import dotenv
dotenv.load_dotenv()
from kbel.disambiguators import Disambiguator
from kbel.core.mention import Mention
from kbel.core.mention import EntityType
from kbel.core.kbel import KBEL
# import logging
# logging.basicConfig(level=logging.DEBUG)

### Creating a KBEL instance (without disambiguator)

In [2]:
kbel = KBEL('wikidata')

#### Getting candidates to a mention:

In [3]:
results = kbel.candidates_lookup(Mention(term='rock', text='', entity_type=EntityType.ITEM), limit=5)
display (*results)

Candidate(iri='http://www.wikidata.org/entity/Q11399', label='rock music', id='http://www.wikidata.org/entity/Q11399', description='popular music genre', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q22731', label='stone', id='http://www.wikidata.org/entity/Q22731', description='rock; building material', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q21003018', label='Rock', id='http://www.wikidata.org/entity/Q21003018', description='family name', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q1851974', label='Rock', id='http://www.wikidata.org/entity/Q1851974', description='village in Worcestershire, England, United Kingdom', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q1404150', label='rock', id='http://www.wikidata.org/entity/Q1404150', description='mass of stone projecting out of the ground or water', aliases=[], metadata={})

### Using `naive` disambiguator to link entities from Wikidata

In [4]:
from kbel.disambiguators.naive import NaiveDisambiguator

kbel = KBEL('wikidata', disambiguator=Disambiguator('naive'))

results = kbel.link(
    mention=Mention(term='rock', text='', entity_type=EntityType.ITEM))
display (*results)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q21003018')), score=None)

### Using `similarity` disambiguator to link entities from Wikidata

In [5]:
# pip install "kbel[similarity]"
from kbel.disambiguators.similarity import SimilarityDisambiguator
kbel.disambiguator = Disambiguator('sim')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
results = kbel.link(
    mention=Mention(term='Rock', text='Rock is a stone', entity_type=EntityType.ITEM),
    limit=100)
display (*results)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q22731')), score=0.7655999660491943)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q60589667')), score=0.618131697177887)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q1404150')), score=0.6179788112640381)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q21003018')), score=0.5909359455108643)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q11399')), score=0.5779575109481812)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q7749')), score=0.5327057838439941)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q42195')), score=0.5188243389129639)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q1851974')), score=0.49860239028930664)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q8063')), score=0.4910997748374939)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q327022')), score=0.3218531608581543)

##### Linking a property instead of an Item

In [7]:
results = kbel.link(
    mention=Mention(term='instance of', text='Rock is a stone', entity_type=EntityType.PROPERTY), limit=1
)

display(*results)

LinkResult(entity=Property(IRI('http://www.wikidata.org/entity/P31'), None), score=0.37770891189575195)

### Using `LLM` disambiguator to link entities from Wikidata

Instantiating LLM Disambiguator with OpenAI's models

In [8]:
# pip install "kbel[llm]"
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-5.2', api_key=os.environ['LLM_API_KEY'])

from kbel.disambiguators.llm import LLM_Disambiguator
kbel.disambiguator = Disambiguator('llm', model= model)
kbel.knowledge_base='wikidata'

In [9]:
results = kbel.link(
    mention=Mention(term='Rock', text='A rock can be used in construction to mimic the appearance and durability of natural stone.', 
                    entity_type=EntityType.ITEM), limit=2)

display (*results)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q8063')), score=None)

##### Adding context to improve disambiguation:

In [10]:
results = kbel.link(
    Mention(
        term="Python",
        text="Python is used for coding.",
        entity_type=EntityType.ITEM,
        context="""
        Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
        """
    )
)

display(*results)

LinkResult(entity=Item(IRI('http://www.wikidata.org/entity/Q28865')), score=None)

### Changing the knowledge source, e.g., DBpedia

In [11]:
kbel.knowledge_base = 'dbpedia'
results = kbel.link(
    Mention(
        term="Python",
        text="Python is used for coding.",
        entity_type=EntityType.ITEM,
        context="""
        Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
        """
    )
)

display(*results)

LinkResult(entity=Item(IRI('http://dbpedia.org/resource/Python_(programming_language)')), score=None)